# 07 — Strings and text

**Workload:** `StringDType` arrays and vectorized `np.strings` cleanup.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Create variable-width string data

Store inconsistent names with NumPy's `StringDType`.

In [2]:
string_dtype = np.dtypes.StringDType()
raw = np.array(["  ADA_Lovelace ", "grace_HOPPER", " Linus_torvalds  "], dtype=string_dtype)
print("raw names:", raw)
print("dtype:", raw.dtype)

raw names: ['  ADA_Lovelace ' 'grace_HOPPER' ' Linus_torvalds  ']
dtype: StringDType()


## Clean text with vectorized operations

Strip whitespace, normalize case and separators, then compute string metadata.

In [3]:
stripped = np.strings.strip(raw)
lowered = np.strings.lower(stripped)
spaced = np.strings.replace(lowered, "_", " ")
cleaned = np.strings.title(spaced)
lengths = np.strings.str_len(cleaned)
separator_positions = np.strings.find(cleaned, " ")
print("cleaned names:", cleaned)
print("lengths:", lengths)
print("separator positions:", separator_positions)

cleaned names: ['Ada Lovelace' 'Grace Hopper' 'Linus Torvalds']
lengths: [12 12 14]
separator positions: [3 5 5]


## Verify the result

In [4]:
assert np.array_equal(cleaned, ["Ada Lovelace", "Grace Hopper", "Linus Torvalds"])
assert np.array_equal(lengths, [12, 12, 14])
assert np.array_equal(separator_positions, [3, 5, 5])
print("PASS — all string assertions passed.")

PASS — all string assertions passed.
